In [2]:
%load_ext autoreload
%autoreload 2

In [64]:
from pathlib import Path
from src.brain_image.model import get_belong_group
from torch import nn
import torch

def N(x):
    return torch.nn.functional.normalize(x, p=2, dim=-1)


def R(x, n=3):
    return torch.round(x * 10**n) / 10**n

paths = [Path("a"), Path("b"), Path("c"), Path("a"), Path("d")]


# canonical axes
e1 = torch.tensor([1.,0.,0.]); e2 = torch.tensor([0.,1.,0.]); e3 = torch.tensor([0.,0.,1.])

v1 = N(torch.stack([
    e1,        # a
    e2,        # b
    e3,        # c
    e1,        # a (duplicate)
    e2,        # d  <-- note: same direction as b
]))

v2 = N(torch.stack([
    e1,        # a
    e2,        # b
    e3,        # c
    e1,        # a
    -e2,       # d  <-- POSITIVE SHOULD BE -1 SIM HERE
]))


x = v1 @ v2.T
l = get_belong_group(paths).float()  # Create multihot labels for each path in batch [[]]
l = l / l.sum(dim=-1)

print((x-0.5)*3)
#clip_loss = nn.CrossEntropyLoss(reduction="none")
loss = nn.functional.cross_entropy(x, l, reduction="none")

print("l", l)
print("x:", R(x))
print("Cross entropy loss", R(loss, n=2))
print("Cross entropy mean", loss.mean(dim=-1), "cosine", x.diag().mean())

tensor([[ 1.5000, -1.5000, -1.5000,  1.5000, -1.5000],
        [-1.5000,  1.5000, -1.5000, -1.5000, -4.5000],
        [-1.5000, -1.5000,  1.5000, -1.5000, -1.5000],
        [ 1.5000, -1.5000, -1.5000,  1.5000, -1.5000],
        [-1.5000,  1.5000, -1.5000, -1.5000, -4.5000]])
l tensor([[0.5000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000]])
x: tensor([[ 1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  0.,  0., -1.],
        [ 0.,  0.,  1.,  0.,  0.],
        [ 1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  0.,  0., -1.]])
Cross entropy loss tensor([1.1300, 0.8100, 0.9000, 1.1300, 2.8100])
Cross entropy mean tensor(1.3564) cosine tensor(0.6000)


In [49]:
v1 = N(torch.stack([
    e1,                 # a
    e2,                 # b
    e3,                 # c
    e1,                 # a
    e2,                 # d
]))

# Make a hard negative near e1 but labeled as 'b'
hard_neg = N(torch.tensor([0.98, 0.20, 0.0]))  # ~0.98 cosine with e1

v2 = N(torch.stack([
    e1,        # a  (true positive for rows a)
    hard_neg,  # b  (but very close to e1 -> hard negative for a-rows)
    e3,        # c
    e1,        # a
    e2,        # d
]))


x = v1 @ v2.T
l = get_belong_group(paths)  # Create multihot labels for each path in batch [[]]

loss = nn.functional.binary_cross_entropy_with_logits((x-0.5)*torch.exp(torch.tensor(5)), l.float(), reduction="none")

print("x:", R(x))
print("Cross entropy loss", R(loss, n=2))
print("Cross entropy mean", loss.mean(dim=-1), "cosine", x.diag().mean())

x: tensor([[1.0000, 0.9800, 0.0000, 1.0000, 0.0000],
        [0.0000, 0.2000, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000],
        [1.0000, 0.9800, 0.0000, 1.0000, 0.0000],
        [0.0000, 0.2000, 0.0000, 0.0000, 1.0000]])
Cross entropy loss tensor([[ 0.0000, 71.2100,  0.0000,  0.0000,  0.0000],
        [ 0.0000, 44.5300,  0.0000,  0.0000, 74.2100],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, 71.2100,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])
Cross entropy mean tensor([1.4242e+01, 2.3747e+01, 1.1845e-33, 1.4242e+01, 1.1845e-33]) cosine tensor(0.8400)


In [5]:
m = N(torch.tensor([1., 1., 0.]))

v1 = torch.stack([m, m, m, m, m])        # collapsed queries
v2 = N(torch.tensor([
    [ 1.,  1.,  0.],    # near m
    [ 1., -1.,  0.],    # orthogonal-ish to m
    [ 0.,  0.,  1.],    # orthogonal to m
    [ 1.,  1.,  0.],    # near m
    [-1., -1.,  0.],    # opposite of m
], dtype=torch.float))

x = v1 @ v2.T
l = get_belong_group(paths)  # Create multihot labels for each path in batch [[]]

loss = nn.functional.binary_cross_entropy_with_logits(x*25, l, reduction="none")

print("x:", R(x))
print("Cross entropy loss", R(loss, n=2))
print("Cross entropy mean", loss.mean(dim=-1), "cosine", x.diag().mean())

x: tensor([[ 1.,  0.,  0.,  1., -1.],
        [ 1.,  0.,  0.,  1., -1.],
        [ 1.,  0.,  0.,  1., -1.],
        [ 1.,  0.,  0.,  1., -1.],
        [ 1.,  0.,  0.,  1., -1.]])
Cross entropy loss tensor([[ 0.0000,  0.6900,  0.6900,  0.0000,  0.0000],
        [25.0000,  0.6900,  0.6900, 25.0000,  0.0000],
        [25.0000,  0.6900,  0.6900, 25.0000,  0.0000],
        [ 0.0000,  0.6900,  0.6900,  0.0000,  0.0000],
        [25.0000,  0.6900,  0.6900, 25.0000, 25.0000]])
Cross entropy mean tensor([ 0.2773, 10.2773, 10.2773,  0.2773, 15.2773]) cosine tensor(0.2000)


In [6]:
a_core = N(torch.tensor([1.,  1., 0.]))
b_core = N(torch.tensor([1., -1., 0.]))
c_core = N(torch.tensor([0.,  0., 1.]))

v1 = N(torch.stack([
    a_core,                          # a
    b_core,                          # b
    c_core,                          # c
    N(a_core + 0.05*torch.tensor([0.,0.,1.])),  # a (tight around a_core)
    N(b_core + 0.05*torch.tensor([0.,0.,1.])),  # d (intentionally near b space)
]))

v2 = N(torch.stack([
    a_core,                          # a
    b_core,                          # b
    c_core,                          # c
    a_core,                          # a
    b_core,                          # d ~ b space
]))

x = v1 @ v2.T
l = get_belong_group(paths)  # Create multihot labels for each path in batch [[]]

loss = nn.functional.binary_cross_entropy_with_logits(x*25, l, reduction="none")

print("x:", R(x))
print("Cross entropy loss", R(loss, n=2))
print("Cross entropy mean", loss.mean(dim=-1), "cosine", x.diag().mean())

x: tensor([[1.0000, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000],
        [0.9990, 0.0000, 0.0500, 0.9990, 0.0000],
        [0.0000, 0.9990, 0.0500, 0.0000, 0.9990]])
Cross entropy loss tensor([[ 0.0000,  0.6900,  0.6900,  0.0000,  0.6900],
        [ 0.6900,  0.0000,  0.6900,  0.6900, 25.0000],
        [ 0.6900,  0.6900,  0.0000,  0.6900,  0.6900],
        [ 0.0000,  0.6900,  1.5000,  0.0000,  0.6900],
        [ 0.6900, 24.9700,  1.5000,  0.6900,  0.0000]])
Cross entropy mean tensor([0.4159, 5.4159, 0.5545, 0.5774, 5.5712]) cosine tensor(0.9995)


In [7]:
e1 = torch.tensor([1.,0.,0.]); e2 = torch.tensor([0.,1.,0.]); e3 = torch.tensor([0.,0.,1.])

# Make row 2 (c) an outlier anchor in a weird direction
outlier = N(torch.tensor([0.3, 0.1, 0.95]))

v1 = N(torch.stack([
    e1,            # a
    e2,            # b
    outlier,       # c (outlier anchor)
    e1,            # a
    e2,            # d
]))

# "c" positive isn’t great; a negative aligns better
weak_c = N(torch.tensor([0.2, 0.1, 0.97]))     # ~0.99 with outlier
better_neg = N(torch.tensor([0.28, 0.12, 0.95])) # ~even closer but labeled as 'b'

v2 = N(torch.stack([
    e1,           # a
    better_neg,   # b  <-- hard negative for c
    weak_c,       # c  <-- true positive for c
    e1,           # a
    e2,           # d
]))

x = v1 @ v2.T
l = get_belong_group(paths)  # Create multihot labels for each path in batch [[]]

loss = nn.functional.binary_cross_entropy_with_logits(x*25, l, reduction="none")

print("x:", R(x))
print("Cross entropy loss", R(loss, n=2))
print("Cross entropy mean", loss.mean(dim=-1), "cosine", x.diag().mean())

x: tensor([[1.0000, 0.2810, 0.2010, 1.0000, 0.0000],
        [0.0000, 0.1200, 0.1000, 0.0000, 1.0000],
        [0.3000, 1.0000, 0.9950, 0.3000, 0.1000],
        [1.0000, 0.2810, 0.2010, 1.0000, 0.0000],
        [0.0000, 0.1200, 0.1000, 0.0000, 1.0000]])
Cross entropy loss tensor([[ 0.0000,  7.0200,  5.0300,  0.0000,  0.6900],
        [ 0.6900,  0.0500,  2.5900,  0.6900, 25.0000],
        [ 7.4900, 24.9900,  0.0000,  7.4900,  2.5800],
        [ 0.0000,  7.0200,  5.0300,  0.0000,  0.6900],
        [ 0.6900,  3.0600,  2.5900,  0.6900,  0.0000]])
Cross entropy mean tensor([2.5480, 5.8048, 8.5097, 2.5480, 1.4062]) cosine tensor(0.8230)


## InfoNCE

In [ ]:



paths = [Path("a"), Path("b"), Path("c"), Path("a"), Path("d")]


# canonical axes
e1 = torch.tensor([1.,0.,0.]); e2 = torch.tensor([0.,1.,0.]); e3 = torch.tensor([0.,0.,1.])

v1 = N(torch.stack([
    e1,        # a
    e2,        # b
    e3,        # c
    e1,        # a (duplicate)
    e2,        # d  <-- note: same direction as b
]))

v2 = N(torch.stack([
    e1,        # a
    e2,        # b
    e3,        # c
    e1,        # a
    -e2,       # d  <-- POSITIVE SHOULD BE -1 SIM HERE
]))


l = get_belong_group(paths, to_float=False)  # Create multihot labels for each path in batch [[]]

nce_loss = InfoNCELoss()
loss, x = nce_loss(v1, v2, l, reduce=False, symmetric=False)

print(l)
print("x1", R(v1))
print("x2", R(v2))
print("x:", R(x))
print("InfoNCE loss", R(loss, n=2))
print("InfoNCE loss mean", loss.mean(dim=-1), "cosine", x.diag().mean())


tensor([[ True, False, False,  True, False],
        [False,  True, False, False, False],
        [False, False,  True, False, False],
        [ True, False, False,  True, False],
        [False, False, False, False,  True]])
x1 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]])
x2 tensor([[ 1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  0.],
        [-0., -1., -0.]])
x: tensor([[ 1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  0.,  0., -1.],
        [ 0.,  0.,  1.,  0.,  0.],
        [ 1.,  0.,  0.,  1.,  0.],
        [ 0.,  1.,  0.,  0., -1.]])
InfoNCE loss tensor([ -0.,  -0.,  -0.,  -0., 200.], grad_fn=<DivBackward0>)
InfoNCE loss mean tensor(40., grad_fn=<MeanBackward1>) cosine tensor(0.6000)
